In [1]:
using ITensors, TimeEvoMPS

In [2]:
using LinearAlgebra

In [3]:
using DelimitedFiles

In [4]:
using ForwardDiff

In [5]:
ITensors.state(::StateName"Up", ::SiteType"S=3/2")  = [1.0, 0.0, 0.0, 0.0]

In [6]:
ITensors.state(::StateName"Dn", ::SiteType"S=3/2")  = [0.0, 0.0, 0.0, 1.0]

In [7]:
ITensors.state(::StateName"Up1", ::SiteType"S=3/2") = [0.0, 1.0, 0.0, 0.0]

In [8]:
ITensors.state(::StateName"Dn1", ::SiteType"S=3/2") = [0.0, 0.0, 1.0, 0.0]

In [9]:
function ITensors.op!(Op::ITensor,
                      ::OpName"Sz",
                      ::SiteType"S=3/2",
                      s::Index)
  Op[s'=>1,s=>1] = +3/2
  Op[s'=>2,s=>2] = +1/2
  Op[s'=>3,s=>3] = -1/2
  Op[s'=>4,s=>4] = -3/2
end        

In [10]:
function ITensors.op!(Op::ITensor,
  ::OpName"Id",
  ::SiteType"S=3/2",
  s::Index)
Op[s'=>1,s=>1] = 1
Op[s'=>2,s=>2] = 1
Op[s'=>3,s=>3] = 1
Op[s'=>4,s=>4] = 1
end

In [11]:
function ITensors.space(::SiteType"S=3/2"; conserve_qns=false)
  if conserve_qns
    return [QN("Sz",3)=>1,QN("Sz",1)=>1, QN("Sz",-1)=>1,QN("Sz",-3)=>1]
  end

return 4
end

In [12]:
function Set_Hamiltonian(N,J_1,J_2,D,E,h0,sites)::MPO
  println(J_1,J_2)
  ampo = OpSum()
  for j=1:N-1
    ampo += 0.5*J_1,"S+",j,"S-",j+1
    ampo += 0.5*J_1,"S-",j,"S+",j+1
    ampo += J_1,"Sz",j,"Sz",j+1
  end

  for j=1:N-2
    ampo += 0.5*J_2,"S+",j,"S-",j+2
    ampo += 0.5*J_2,"S-",j,"S+",j+2
    ampo += J_2,"Sz",j,"Sz",j+2  
  end

  for j=1:N
    ampo += D,"Sz",j,"Sz",j

    # Sx
    ampo += 0.5*E,"S+",j
    ampo += 0.5*E,"S-",j
    #Sy
    ampo += -0.5*1im*E,"S+",j
    ampo += +0.5*1im*E,"S+",j

    ######### field

    ampo += h0*j,"Sz",j

  end
  
  HH=MPO(ampo,sites)
  return HH
end

Set_Hamiltonian (generic function with 1 method)

In [13]:
######################## Imbalance #####################

function Imbalance(M1,M2,N)
  S=0
  for i=1:N 
    S+=M1[i,i]*M2[i,i]
  end
  return real(4*S/(9*N))
end

Imbalance (generic function with 1 method)

In [14]:
function VN_entropy(psi::MPS, b::Int)
  s = siteinds(psi)  
  orthogonalize!(psi, b)
  _,S = svd(psi[b], (linkind(psi, b-1), s[b]))
  SvN = 0.0
  for n in 1:dim(S, 1)
    p = S[n,n]^2
    SvN -= p * log(p)
  end
  return SvN
end

VN_entropy (generic function with 1 method)

In [15]:
ITensors.space(::SiteType"S=1/2") = 2

In [22]:
theta_g    = [0,15,45,60,180,210,300]
for θ in theta_g
    println("Valor: ", θ)
end

Valor: 0
Valor: 15
Valor: 45
Valor: 60
Valor: 180
Valor: 210
Valor: 300


In [25]:
let
    N = 12;  @show N
    J = 1.0; D = 0; E = 0
    p = Int(N/2 + 1)
    @show p
    L = N/2; maxdim = 256; h0 = 0
    theta_g    = [0,15,45,60,180,210,300]
    theta_g = 0:1:360

    # arquivo único de saída
    path = "data/r_N12.dat"

    # se quiser começar com arquivo vazio a cada execução:
    open(path, "w") do io
        println(io, "angulo_deg;angulo_rad;energia;entropia")  # cabeçalho opcional
    end


    nsweeps = 10
    ek = nsweeps

    sites = siteinds("S=1/2", N; conserve_qns=true, conserve_sz=false)
    sweeps = Sweeps(nsweeps)
    maxdim!(sweeps, 50, 100, 200, 400)
    cutoff!(sweeps, 1E-10)

    # estados iniciais
    state1 = [isodd(n) ? "Up" : "Dn" for n in 1:N]
    psi1 = randomMPS(sites, state1; linkdims=ek)

    state2 = [isodd(n) ? "Dn" : "Up" for n in 1:N]
    psi2 = randomMPS(sites, state2; linkdims=ek)

    psi0 = +(psi1, psi2)
    println(psi0)
    normalize!(psi0)

    # loop sobre os ângulos
    for (idx, ang) in enumerate(theta_g)
        theta = deg2rad(ang)
        J_1 = J * cos(theta)
        J_2 = J * sin(theta)
        println("Ângulo (graus): $ang  ->  radianos: $theta")

        @show h0
        H = Set_Hamiltonian(N, J_1, J_2, D, E, h0, sites)
        energy_base, psi_base = dmrg(H, psi0; nsweeps, eigsolve_krylovdim=3)
        entropy = VN_entropy(psi_base, p)

        # monta string de dados
        data = string(ang, ";", theta, ";", energy_base, ";", entropy)

        open(path, "a") do io
            println(io, data)
        end
    end
end

N = 12
p = 7
MPS
[1] ((dim=2|id=339|"S=1/2,Site,n=1"), (dim=4|id=59|"Link,l=1"))
[2] ((dim=2|id=642|"S=1/2,Site,n=2"), (dim=8|id=448|"Link,l=2"), (dim=4|id=59|"Link,l=1"))
[3] ((dim=2|id=503|"S=1/2,Site,n=3"), (dim=16|id=269|"Link,l=3"), (dim=8|id=448|"Link,l=2"))
[4] ((dim=2|id=313|"S=1/2,Site,n=4"), (dim=20|id=132|"Link,l=4"), (dim=16|id=269|"Link,l=3"))
[5] ((dim=2|id=808|"S=1/2,Site,n=5"), (dim=20|id=685|"Link,l=5"), (dim=20|id=132|"Link,l=4"))
[6] ((dim=2|id=1|"S=1/2,Site,n=6"), (dim=20|id=697|"Link,l=6"), (dim=20|id=685|"Link,l=5"))
[7] ((dim=2|id=539|"S=1/2,Site,n=7"), (dim=20|id=95|"Link,l=7"), (dim=20|id=697|"Link,l=6"))
[8] ((dim=2|id=913|"S=1/2,Site,n=8"), (dim=16|id=116|"Link,l=8"), (dim=20|id=95|"Link,l=7"))
[9] ((dim=2|id=730|"S=1/2,Site,n=9"), (dim=8|id=795|"Link,l=9"), (dim=16|id=116|"Link,l=8"))
[10] ((dim=2|id=421|"S=1/2,Site,n=10"), (dim=4|id=188|"Link,l=10"), (dim=8|id=795|"Link,l=9"))
[11] ((dim=2|id=435|"S=1/2,Site,n=11"), (dim=2|id=855|"Link,l=11"), (dim=4|id=188